# Pydantic AI with Gemini — typed output in a notebook

Same agent as `agent_app.py`, with the one change a notebook forces: **`run_sync()` does not work
here.** The kernel is already running an asyncio event loop, so this notebook uses top-level
`await` instead.

Needs `GEMINI_API_KEY` in a `.env` file beside this notebook.

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # before the agent is built — the provider reads the key from the environment

from pydantic import BaseModel, Field
from pydantic_ai import Agent, ModelRetry


class DatabaseQuery(BaseModel):
    sql_query: str = Field(description="Valid PostgreSQL query")
    explanation: str = Field(description="Brief breakdown of what the query does")


agent = Agent(
    "google:gemini-3.6-flash",
    output_type=DatabaseQuery,
    instructions=(
        "You write PostgreSQL against a schema with users(id, name) and "
        "orders(user_id, total_amount, created_at). Answer with the query itself, "
        "never with a schema-inspection query."
    ),
    retries=3,
)
print("agent ready")

## The trap: `run_sync()` in a notebook

Worth running once so the error is familiar. This cell is *supposed* to fail.

In [ ]:
try:
    agent.run_sync("Select all users.")
except RuntimeError as e:
    print("RuntimeError:", e)

## The fix: top-level `await`

`agent.run(...)` is the coroutine `run_sync` wraps. IPython allows `await` at cell level, so the
notebook's existing loop runs it.

In [ ]:
result = await agent.run("Find the top 5 users by total spend in 2026")
print(result.output.sql_query)

The return value is a real `DatabaseQuery`, not a string — the notebook renders it as the
model it is.

In [ ]:
result.output

## The validator, and the retry it causes

The schema cannot express "this query has a LIMIT". A validator can, and raising `ModelRetry`
sends the complaint back to the model as a new turn.

In [ ]:
attempts = []


@agent.output_validator
def must_be_capped(out: DatabaseQuery) -> DatabaseQuery:
    attempts.append(out.sql_query)
    if "limit" not in out.sql_query.lower():
        raise ModelRetry("The query must include an explicit LIMIT clause. Add one.")
    return out


capped = await agent.run("Show every user's total spend in 2026, ordered by spend.")

for i, a in enumerate(attempts, 1):
    print(f"attempt {i}: {'LIMIT' if 'limit' in a.lower() else 'NO LIMIT'}")
print("requests billed:", capped.usage.requests)

`requests` is the honest cost of the retry: one prompt, more than one round trip.